In [1]:
pip install optuna asgl

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 20.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 53.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 65.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 131.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 165.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 123

In [13]:
import numpy as np
import pandas as pd

import time
import warnings

import optuna
import json
import os

from asgl import Regressor
import random
random.seed(42)
np.random.seed(42)
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('time_series_15min_cleaned.csv')

a = np.array(df.values[:, 2])
a = np.expand_dims(a, axis=1)

def split_data(data, horizon, window):
    n, m = data.shape
    X = np.zeros((n - window - horizon, window))
    Y = np.zeros((n - window - horizon,1))

    for i in range(n - window - horizon):
        start = i
        end = start + window

        X[i, :] = data[start:end,0]
        Y[i] = data[end + horizon - 1,0]

    return X, Y

X,Y = split_data(a,1,96)
n1=X.shape[0]

train_X, train_Y = X[:round(0.8 * n1)], Y[:round(0.8 * n1)]
val_X, val_Y = X[round(0.8 * n1):round(0.9 * n1)], Y[round(0.8 * n1):round(0.9 * n1)]
test_X, test_Y = X[round(0.9 * n1):], Y[round(0.9 * n1):]

tau_values = [round(i, 2) for i in np.arange(0.05, 1.0, 0.05)]

In [4]:
def pinball_loss(y_true, y_pred, tau):
    """
    Compute pinball loss for a single quantile.
    """
    errors = y_true - y_pred
    loss = np.where(errors >= 0, tau * errors, (tau - 1) * errors)
    return np.mean(loss)


def multi_quantile_pinball_loss(y_true, y_pred_all, tau_values):
    """
    Compute average pinball loss across multiple quantiles.
    """
    total_loss = 0
    n_quantiles = len(tau_values)

    for i, tau in enumerate(tau_values):
        total_loss += pinball_loss(y_true, y_pred_all[:, i], tau)

    return total_loss / n_quantiles

In [5]:
def objective(trial):
    """
    Optuna objective function for hyperparameter optimization.
    """
    # Define lambda search range (can be adjusted after standardization)
    lambda_val = trial.suggest_float('lambda', 0.001, 10)

    # Store validation set predictions for each quantile
    predictions_val = []

    start_time = time.time()

    for tau in tau_values:
        # Create model
        model = Regressor(model='qr', penalization='lasso', quantile=tau, lambda1=lambda_val)

        # Fit model (using standardized X)
        model.fit(train_X, train_Y)

        # Predict on validation set (using standardized X)
        y_pred_val = model.predict(val_X).reshape(-1, 1)
        predictions_val.append(y_pred_val)

    # Combine predictions for all quantiles
    predictions_val_matrix = np.hstack(predictions_val)

    # Compute average pinball loss
    current_loss = multi_quantile_pinball_loss(val_Y, predictions_val_matrix, tau_values)

    trial_time = time.time() - start_time
    trial.set_user_attr('time', trial_time)

    return current_loss

Optuna searching (Option C)

In [6]:
def optimize_lambda(n_trials=10):
    """
    Optimize lambda parameter using Optuna.
    """
    print(f"Starting Optuna optimization with {n_trials} trials...")

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    start_time = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    total_time = time.time() - start_time

    print(f"\nOptimization complete. Total time: {total_time:.3f} seconds")
    print(f"Best lambda: {study.best_params['lambda']:.6f}")
    print(f"Best validation loss: {study.best_value:.6f}")

    results = {
    'best_lambda': study.best_params['lambda'],
    'best_loss': study.best_value,
    'best_trial_number': study.best_trial.number,
    'total_time': total_time,
    'study': {
        'study_name': study.study_name,
        'direction': str(study.direction),
        'trials': [
            {
                'number': t.number,
                'params': t.params,
                'value': t.value,
                'state': str(t.state),
                'datetime_start': str(t.datetime_start),
                'datetime_complete': str(t.datetime_complete),
            }
            for t in study.trials
        ]
    }
}

    with open("optuna_lasso_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    return results

#Run for finding lambada
optuna_results = optimize_lambda(n_trials=30)
best_lambda=optuna_results['best_lambda']

[I 2026-05-05 16:19:53,267] A new study created in memory with name: no-name-ba815582-4faf-4ada-80e0-af789bde05d2


Starting Optuna optimization with 1 trials...


Best trial: 0. Best value: 894.204: 100%|██████████| 1/1 [13:27<00:00, 807.86s/it]

[I 2026-05-05 16:33:21,122] Trial 0 finished with value: 894.2036238161639 and parameters: {'lambda': 3.7460266483547775}. Best is trial 0 with value: 894.2036238161639.

Optimization complete. Total time: 807.857 seconds
Best lambda: 3.746027
Best validation loss: 894.203624


Using hyperparameters directly, without optuna (Option B)

In [5]:
with open('optuna_lasso_results.json', 'r') as f:
    results = json.load(f)

best_lambda = results['best_lambda']
print("Best lambda:", best_lambda)

Best lambda: 9.909733546017952


Test set evaluation

In [7]:
# Evaluate on the test set using the best lambda.
def evaluate_on_test_set(lambda_val):

    print(f"\nEvaluating on test set with lambda={lambda_val:.6f}")

    test_predictions = []

    start_time = time.time()

    for tau in tau_values:
        model = Regressor(model='qr', penalization='lasso', quantile=tau, lambda1=lambda_val)

        model.fit(train_X, train_Y)
        y_pred_test = model.predict(test_X).reshape(-1, 1)
        test_predictions.append(y_pred_test)

    test_predictions_matrix = np.hstack(test_predictions)

    # Compute average pinball loss on test set
    test_loss = multi_quantile_pinball_loss(test_Y, test_predictions_matrix, tau_values)

    eval_time = time.time() - start_time

    print(f"\n{'=' * 60}")
    print(f"Test set evaluation results:")
    print(f"Lambda used: {lambda_val:.6f}")
    print(f"Test set average pinball loss: {test_loss:.3f}")
    print(f"Evaluation time: {eval_time:.3f} seconds")
    print(f"{'=' * 60}")

    json_path = "optuna_lasso_results.json"
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            results = json.load(f)
    else:
        results = {}

    results["eval_time"] = eval_time

    with open(json_path, "w") as f:
        json.dump(results, f, indent=2)

    return test_loss, test_predictions_matrix

In [14]:
test_loss, test_predictions_matrix = evaluate_on_test_set(best_lambda)


Evaluating on test set with lambda=3.746027

Test set evaluation results:
Lambda used: 3.746027
Test set average pinball loss: 920.692
Evaluation time: 844.638 seconds


In [12]:
columns = [f'q{int(tau * 100)}' for tau in tau_values]
all_pred_df = pd.DataFrame(test_predictions_matrix, columns=columns)
all_pred_df.to_excel('lassopred.xlsx', index=False)

In [13]:
all_pred_df

,q5,q10,q15,q20,q25,q30,q35,q40,q45,q50,q55,q60,q65,q70,q75,q80,q85,q90,q95
0,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
1,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
2,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
3,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
4,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
1138,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
1139,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
1140,-0.310128,0.205342,1.289874,2.639963,4.199869,5.909996,7.500842,8.899923,10.550243,12.189979,14.010023,15.680613,17.350133,19.440044,21.615531,24.030022,26.620795,29.180955,32.887342
